# StormEngine V7-A workflow

239 physical stations, 12-hour history, five mask-aware inputs (`u10`, `v10`, `i10fg`, `t2m`, `tp`), and six forecast hours. Run one stage at a time. Formal training is disabled by default.

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
CONFIG = 'configs/v7_a.yaml'
DEVICE = 'cuda'
RUN_FORMAL_TRAINING = False
RUN_FULL_2017_EVALUATION = False
RUN_DPC_REPLAY = False
SAVE_DPC_PREDICTIONS = False

def run(*args):
    command = [sys.executable, *map(str, args)]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)

print(ROOT)

## 1. Unit tests and preflight

In [ ]:
run('-m', 'pytest', 'tests/test_v7_dataset.py', 'tests/test_v7_input.py', 'tests/test_v7_model.py', '-q')
run('scripts/check_v7.py', 'preflight', '--config', CONFIG, '--device', DEVICE)

## 2. Smoke test and 200-batch benchmark

In [ ]:
run('scripts/check_v7.py', 'smoke', '--config', CONFIG, '--device', DEVICE)
run('scripts/check_v7.py', 'benchmark', '--config', CONFIG, '--device', DEVICE, '--batches', '200')

## 3. Short pilot (5 epochs, capped batches)

In [ ]:
run('scripts/check_v7.py', 'pilot', '--config', CONFIG, '--device', DEVICE, '--epochs', '5')

## 4. Formal 2010–2015 training
Set `RUN_FORMAL_TRAINING=True` only after reviewing pilot loss and speed. Saves `best.pt` and `last.pt`; use `--resume artifacts/v7_a_2010_2017/last.pt` to continue an interrupted run.

In [ ]:
if RUN_FORMAL_TRAINING:
    run('scripts/check_v7.py', 'train', '--config', CONFIG, '--device', DEVICE)
else:
    print('Formal training skipped. Set RUN_FORMAL_TRAINING=True when ready.')

## 5. Frozen 2017 evaluation
Runs clean input plus missing-input seeds. Each result includes V7-A, dense persistence, and input-fair sparse IDW metrics for full/land/sea and leads +1…+6. Compare the clean result with the frozen V6 JSON already stored under `results/v6_2010_2017_baseline`. Do not use 2017 to select the model.

In [ ]:
if RUN_FULL_2017_EVALUATION:
    run('scripts/evaluate_v7_a.py', '--config', CONFIG, '--scenario', 'clean', '--device', DEVICE)
    for seed in (42, 123, 2026):
        run('scripts/evaluate_v7_a.py', '--config', CONFIG, '--scenario', 'missing', '--seed', str(seed), '--device', DEVICE)
else:
    print('2017 evaluation skipped. Set RUN_FULL_2017_EVALUATION=True after training is frozen.')

## 6. Real DPC week replay
This verifies operational compatibility. It is not an accuracy evaluation unless matching 2026 ERA5 targets are added.

In [ ]:
if RUN_DPC_REPLAY:
    args = ['scripts/replay_v7_a_dpc.py', '--config', CONFIG, '--device', DEVICE]
    if SAVE_DPC_PREDICTIONS: args.append('--save-predictions')
    run(*args)
else:
    print('DPC replay skipped. Set RUN_DPC_REPLAY=True after best.pt exists.')

## 7. Freeze V7-A
Run only after formal training, 2017 evaluation, and DPC replay are complete. The manifest records hashes without adding the large checkpoint to Git.

In [ ]:
if RUN_FORMAL_TRAINING and RUN_FULL_2017_EVALUATION and RUN_DPC_REPLAY:
    run('scripts/freeze_v7_a.py', '--config', CONFIG)
else:
    print('Freeze skipped until all three formal stages are explicitly enabled and completed.')